# J-space / Jacobian Lens bounded Colab smoke test

Purpose: validate that a pinned Qwen3-1.7B model and matching pre-fitted Jacobian lens can produce sparse, provenance-bearing observations on an assigned NVIDIA GPU.

Boundaries:
- Observation only; no lens fitting, steering, ablation, Sakshi/Elume mutation, or external publication.
- Use only the included non-sensitive synthetic prompt for the first run.
- Store the prompt digest and token count, not the raw prompt, in the exported artifact.
- Lens readouts are measurements, not ground truth about beliefs, intentions, consciousness, or cognition.
- Colab hardware is variable. The preflight aborts before model downloads if CUDA or minimum VRAM is unavailable.

In Colab, select **Runtime → Change runtime type → GPU**, then run cells top-to-bottom.


## 0. Pinned source identities

These revisions were directly checked during notebook construction on 2026-07-17. Re-run the source-metadata cell before treating a later run as equivalent.


In [ ]:
JLENS_REPO_URL = "https://github.com/anthropics/jacobian-lens.git"
JLENS_COMMIT = "581d398613e5602a5af361e1c34d3a92ea82ba8e"
MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "a4114d7752d11eb546e6cf372213d7e75526d3a1"
LENS_FILE = "qwen3-1.7b/jlens/Salesforce-wikitext/Qwen3-1.7B_jacobian_lens.pt"
MIN_VRAM_GIB = 14.0
TOP_K = 10
MAX_PROMPT_TOKENS = 128


## 1. Install the pinned reference implementation

The Git commit is pinned. Model and lens artifacts are separately pinned in the configuration cell.


In [ ]:
%pip install -q "git+https://github.com/anthropics/jacobian-lens.git@{JLENS_COMMIT}" "transformers>=5.5" "huggingface_hub>=0.30" "safetensors"


## 2. Fail-fast accelerator and environment preflight

This runs before downloading the model or lens. A failure is a measured capacity result; do not weaken the threshold merely to force the experiment through.


In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch
import transformers
import huggingface_hub

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU unavailable. In Colab select Runtime → Change runtime type → GPU.")

device_index = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device_index)
vram_gib = props.total_memory / (1024 ** 3)
if vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(
        f"Assigned GPU has {vram_gib:.2f} GiB VRAM; this bounded spike requires at least "
        f"{MIN_VRAM_GIB:.1f} GiB. Request another runtime or move to an L4-class VM."
    )

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
disk = shutil.disk_usage("/content" if Path("/content").exists() else ".")
try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
except Exception as exc:
    nvidia_smi = f"unavailable: {type(exc).__name__}: {exc}"

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu_name": props.name,
    "gpu_total_vram_gib": round(vram_gib, 3),
    "compute_capability": f"{props.major}.{props.minor}",
    "compute_dtype": str(compute_dtype),
    "disk_free_gib": round(disk.free / (1024 ** 3), 3),
    "nvidia_smi": nvidia_smi,
}
print(json.dumps(environment, indent=2))


## 3. Resolve and verify exact Hub revisions

This checks that the requested revisions resolve to the expected commits, downloads only the lens file, and computes its SHA-256 checksum before deserialization.


In [ ]:
import hashlib
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
resolved_model_revision = api.model_info(MODEL_ID, revision=MODEL_REVISION).sha
resolved_lens_revision = api.model_info(LENS_REPO, revision=LENS_REVISION).sha
assert resolved_model_revision == MODEL_REVISION, (resolved_model_revision, MODEL_REVISION)
assert resolved_lens_revision == LENS_REVISION, (resolved_lens_revision, LENS_REVISION)

lens_path = hf_hub_download(
    repo_id=LENS_REPO,
    filename=LENS_FILE,
    revision=LENS_REVISION,
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

lens_sha256 = sha256_file(lens_path)
print(json.dumps({
    "model_revision": resolved_model_revision,
    "lens_revision": resolved_lens_revision,
    "lens_file": LENS_FILE,
    "lens_size_bytes": os.path.getsize(lens_path),
    "lens_sha256": lens_sha256,
}, indent=2))


## 4. Load the matching model and lens

The compatibility checks are mandatory: model width, layer range, and pinned identity must agree before observation.


In [ ]:
import jlens
from transformers import AutoModelForCausalLM, AutoTokenizer

jlens.configure_logging()
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=compute_dtype,
    low_cpu_mem_usage=True,
).cuda().eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = jlens.from_hf(hf_model, tokenizer, compile=False)
lens = jlens.JacobianLens.load(lens_path)

assert model.d_model == 2048, f"Unexpected model width: {model.d_model}"
assert model.n_layers == 28, f"Unexpected layer count: {model.n_layers}"
assert lens.d_model == model.d_model, (lens.d_model, model.d_model)
assert lens.source_layers, "Lens contains no fitted source layers"
assert min(lens.source_layers) >= 0 and max(lens.source_layers) < model.n_layers

print(model)
print(lens)
print(f"allocated={torch.cuda.memory_allocated() / (1024**3):.3f} GiB; "
      f"reserved={torch.cuda.memory_reserved() / (1024**3):.3f} GiB")


## 5. Produce a sparse open-loop observation and baseline

The logit lens is a baseline. The Jacobian-lens output is not interpreted as a psychological state. Raw activations and full vocabulary logits are not persisted.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

prompt = "Fact: The currency used in the country shaped like a boot is"
prompt_bytes = prompt.encode("utf-8")
prompt_sha256 = hashlib.sha256(prompt_bytes).hexdigest()
input_ids_for_count = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS).input_ids

source_layers = lens.source_layers
candidate_offsets = [0.25, 0.50, 0.75, 1.0]
layers = sorted({source_layers[min(round((len(source_layers) - 1) * x), len(source_layers) - 1)] for x in candidate_offsets})
positions = [-2]

with torch.inference_mode():
    j_logits_1, model_logits_1, observed_input_ids = lens.apply(
        model, prompt, layers=layers, positions=positions, max_seq_len=MAX_PROMPT_TOKENS
    )
    baseline_logits, _, _ = lens.apply(
        model, prompt, layers=layers, positions=positions,
        max_seq_len=MAX_PROMPT_TOKENS, use_jacobian=False
    )
    j_logits_2, model_logits_2, _ = lens.apply(
        model, prompt, layers=layers, positions=positions, max_seq_len=MAX_PROMPT_TOKENS
    )

def sparse_topk(logits):
    values, indices = logits[0].topk(TOP_K)
    return [
        {"rank": rank, "token_id": int(token_id), "token_text": tokenizer.decode([int(token_id)]), "logit": float(value)}
        for rank, (token_id, value) in enumerate(zip(indices, values), start=1)
    ]

j_readouts = {str(layer): sparse_topk(j_logits_1[layer]) for layer in layers}
baseline_readouts = {str(layer): sparse_topk(baseline_logits[layer]) for layer in layers}
repeatability = {
    "same_topk_token_ids": all(
        j_logits_1[layer][0].topk(TOP_K).indices.equal(j_logits_2[layer][0].topk(TOP_K).indices)
        for layer in layers
    ),
    "max_abs_jacobian_logit_difference": max(
        float((j_logits_1[layer] - j_logits_2[layer]).abs().max()) for layer in layers
    ),
    "max_abs_model_logit_difference": float((model_logits_1 - model_logits_2).abs().max()),
}

observation = {
    "schema": "jspace-observation-smoke-test/v1",
    "run_id": str(uuid4()),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "evidence_class": "direct_runtime_measurement",
    "scope": "open_loop_observation_only",
    "model": {"repo_id": MODEL_ID, "revision": MODEL_REVISION, "n_layers": model.n_layers, "d_model": model.d_model},
    "lens": {
        "repo_id": LENS_REPO, "revision": LENS_REVISION, "filename": LENS_FILE,
        "sha256": lens_sha256, "n_prompts_recorded_in_artifact": lens.n_prompts,
        "source_layers": lens.source_layers, "d_model": lens.d_model,
    },
    "instrumentation": {"repo": JLENS_REPO_URL, "commit": JLENS_COMMIT},
    "input": {
        "sha256": prompt_sha256,
        "utf8_byte_count": len(prompt_bytes),
        "token_count": int(observed_input_ids.shape[-1]),
        "raw_prompt_persisted": False,
        "position_indices": positions,
    },
    "runtime": environment,
    "measurement": {
        "selected_layers": layers,
        "top_k": TOP_K,
        "jacobian_lens": j_readouts,
        "logit_lens_baseline": baseline_readouts,
        "repeatability_same_runtime": repeatability,
        "calibration": "uncalibrated raw logits and ranks",
        "uncertainty": "No construct-level uncertainty model; do not treat readouts as ground truth.",
        "coverage": "Selected fitted layers and one token position only; not full-sequence or full-layer coverage.",
    },
    "retention": {"raw_activations_persisted": False, "full_logits_persisted": False, "raw_prompt_persisted": False},
}

print(json.dumps({
    "run_id": observation["run_id"],
    "layers": layers,
    "repeatability": repeatability,
    "jacobian_top_tokens": {layer: [row["token_text"] for row in rows[:5]] for layer, rows in j_readouts.items()},
}, indent=2, ensure_ascii=False))


## 6. Export a content-addressed observation artifact

The JSON filename is derived from its contents. Existing files are never overwritten. The artifact contains sparse readouts and provenance, not the raw prompt, activations, or full logits.


In [ ]:
artifact_dir = Path("/content/jspace_artifacts" if Path("/content").exists() else "jspace_artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
payload = (json.dumps(observation, sort_keys=True, indent=2, ensure_ascii=False) + "\n").encode("utf-8")
artifact_sha256 = hashlib.sha256(payload).hexdigest()
artifact_path = artifact_dir / f"jspace_observation_{artifact_sha256[:16]}.json"
try:
    with artifact_path.open("xb") as handle:
        handle.write(payload)
except FileExistsError:
    existing_sha256 = sha256_file(artifact_path)
    if existing_sha256 != artifact_sha256:
        raise RuntimeError("Content-addressed artifact path exists with a different checksum")

assert sha256_file(artifact_path) == artifact_sha256
print(json.dumps({
    "artifact_path": str(artifact_path),
    "artifact_size_bytes": artifact_path.stat().st_size,
    "artifact_sha256": artifact_sha256,
}, indent=2))


## 7. Optional explicit download

Run this only if you are authorized to transfer the derived sparse artifact from the Colab runtime to the local computer.


In [ ]:
# Explicitly opt in by changing this to True.
DOWNLOAD_ARTIFACT = False
if DOWNLOAD_ARTIFACT:
    from google.colab import files
    files.download(str(artifact_path))
else:
    print("Artifact retained in the current runtime only:", artifact_path)


## Stage gate

Success here means only that this exact model–lens pair produced reproducible sparse readouts on one recorded runtime. It does not establish construct validity, predictive value, causal value, durable Colab suitability, or readiness for Sakshi/Elume integration.

Next discriminating test: repeat the same pinned prompts across fresh runtimes and compare against output-token/logit, shuffled-layer, and deliberately mismatched-probe controls before any observer integration.
